# 10 – Best PINN run (seed)

Notebooks 03 and 04 can be run for many independent seeds (100 in the paper). This notebook chooses the best run of a case with the score of the **fit**: for every rolling window t0 the forecast file contains the weeks of the window of observations (t ≤ t0), which are compared with the observations (no future week is used), with the weighted interval score (**WIS**) or the **wMAPE** of the median. The score of a seed is the mean over the windows; the best seed is the one with the lowest score.

The score of a window uses the harmonic weights on its weeks (more weight to the most recent ones) when the likelihood of the case uses them (`case.weight`): synthetic scenarios, and real case with the look-back on all the history; not for the real case with look-back 5 weeks. It can be changed with `WEIGHTED`.

The notebook writes `best_seed.json` in `posteriors/pinn/<case>/`, which notebooks 07, 08 and 09 read with `PINN_SEED = "best"`. Run it after notebook 04 (forecasts of all the seeds). Same `CASE` as in the other notebooks. One notebook for the four synthetic scenarios and the real case.

In [ ]:
import os

from pinnvsmech.evaluation.best_seed import rank_seeds, save_best_seed
from pinnvsmech.inference.cases import load_case
from pinnvsmech.paths import OUTPUTS_DIR, PRECOMPUTED_DIR, TABLES_DIR

## Settings

In [ ]:
CASE = "synthetic"                 # "synthetic" or "real"
R0_TRUE = 2.0                      # synthetic only: 1.5 or 2.0
NOISE = "mild"                     # synthetic only: "light" or "mild"
CASE_OVERRIDES = {}                # as in notebooks 03-04 (real: look-back of 5 weeks, no weights, by default)

SOURCE = "outputs"                 # "outputs" (your runs) or "precomputed"
HORIZON = 4                        # forecast files to read (the fit is the same for every horizon)
CRITERION = "WIS"                  # "WIS" or "wMAPE"
WEIGHTED = None                    # None: as the likelihood of the case (case.weight); True / False to force
SAVE = True                        # write best_seed.json and the table of all the seeds

## Score of the fit of every seed

In [ ]:
case = load_case(CASE, R0=R0_TRUE, noise=NOISE, **CASE_OVERRIDES)
root = PRECOMPUTED_DIR if SOURCE == "precomputed" else OUTPUTS_DIR
case_dir = root / "posteriors" / "pinn" / case.label

weighted = case.weight if WEIGHTED is None else WEIGHTED
ranking = rank_seeds(case, case_dir, horizon=HORIZON, criterion=CRITERION, weighted=weighted)

print(case.label, f"| {len(ranking)} seeds | fit {CRITERION}" + (" (weighted)" if weighted else ""))
print(ranking.head(10).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("...")
print(ranking.tail(3).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## Best seed

In [ ]:
best = int(ranking["seed"].iloc[0])
print(f"Best seed: {best}")

if SAVE:
    info = save_best_seed(case_dir, ranking, CRITERION, weighted)
    print("Written", case_dir / "best_seed.json", info)
    table_dir = TABLES_DIR / "forecasts" / case.label
    os.makedirs(table_dir, exist_ok=True)
    ranking.to_csv(table_dir / f"best_seed_fit_{CRITERION}.csv", index=False)